In [1]:
BASE = '/kaggle/input/datasets/bornamuzina/datasettest'
TENSORS = BASE + '/tensors_rgb_packed_k/tensors_rgb_packed'
IMAGES = BASE + '/yolo_rgb_k/yolo_rgb'
print(BASE)

/kaggle/input/datasets/bornamuzina/datasettest


In [2]:
yaml = f"""path: {IMAGES}
train: images/train
val: images/val
test: images/test

names:
  0: drone
"""
open('/kaggle/working/data.yaml','w').write(yaml)
!cat /kaggle/working/data.yaml

path: /kaggle/input/datasets/bornamuzina/datasettest/yolo_rgb_k/yolo_rgb
train: images/train
val: images/val
test: images/test

names:
  0: drone


In [3]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 4.0 MB/s eta 0:00:00


In [4]:
!yolo detect val data=/kaggle/working/data.yaml model={BASE}/yolo26_rgb/runs/yolo26_rgb-2/weights/best.pt imgsz=224 split=test

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.146 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 120 layers, 2,375,031 parameters, 0 gradients, 5.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.4±0.3 ms, read: 2.2±0.2 MB/s, size: 10.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/bornamuzina/datasettest/yolo_rgb_k/yolo_rgb/labels/test... 3723 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3723/3723 295.6it/s 12.6s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/bornamuzina/datasettest/yolo_rgb_k/yolo_rgb/labels is n

In [5]:
!cp {BASE}/*.py /kaggle/working/
!mkdir -p /kaggle/working/runs
!cp -r {BASE}/yolov2_rgb/runs/full_rgb /kaggle/working/runs/
%cd /kaggle/working
!ls runs/full_rgb

/kaggle/working
best.weights.h5  eval_validation.json  history.json


In [6]:
!apt-get install -qq python3.11 python3.11-venv python3.11-dev > /dev/null 2>&1
!python3.11 -m venv /kaggle/working/akv
!/kaggle/working/akv/bin/pip install -q --upgrade pip
!/kaggle/working/akv/bin/pip install -q akida-models==1.14.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [7]:
import re
src = open('config.py').read()
src = re.sub(r"ROOT = Path\(.*?\)", "ROOT = Path('/kaggle/working')", src)
src = re.sub(r"TENSOR_DIR = .*", f"TENSOR_DIR = Path('{TENSORS}')", src)
src = re.sub(r"SPLITS_FILE = .*", f"SPLITS_FILE = Path('{TENSORS}/splits.json')", src)
src = re.sub(r"RUNS_DIR = .*", "RUNS_DIR = Path('/kaggle/working/runs')", src)
open('config.py','w').write(src)

src = open('dataset.py').read()
src = src.replace(
    'if not npy.exists() or not meta_path.exists():',
    'npz = tensor_dir / f"{clip}_tensors.npz"\n    if not meta_path.exists() or (not npy.exists() and not npz.exists()):'
)
src = src.replace(
    'tensors = np.load(npy, mmap_mode="r")',
    'tensors = np.load(npy, mmap_mode="r") if npy.exists() else np.load(npz)["a"]'
)
open('dataset.py','w').write(src)
print(open('config.py').read()[:500])

"""
Every constant that has to agree across the pipeline.

Anchors in particular must be identical in target creation and in
inference decoding. If they drift apart the network trains fine and
then predicts boxes of the wrong size, which looks like a broken model
rather than a broken constant. Keeping them in one file that both sides
import removes that failure mode.
"""

from pathlib import Path


# ============================================================
# PATHS
# =========================


In [8]:
!grep -n "ROOT\|TENSOR_DIR\|SPLITS_FILE\|RUNS_DIR\|CHANNELS" config.py

18:ROOT = Path('/kaggle/working')
19:TENSOR_DIR = Path('/kaggle/input/datasets/bornamuzina/datasettest/tensors_rgb_packed_k/tensors_rgb_packed')
20:# TENSOR_DIR = Path('/kaggle/input/datasets/bornamuzina/datasettest/tensors_rgb_packed_k/tensors_rgb_packed')
21:SPLITS_FILE = Path('/kaggle/input/datasets/bornamuzina/datasettest/tensors_rgb_packed_k/tensors_rgb_packed/splits.json')
22:RUNS_DIR = Path('/kaggle/working/runs')
30:CHANNELS = 3              # signed accumulation; 2 will not convert to Akida - now RGB
31:# CHANNELS = 1


In [9]:
!sed -n '/def to_uint8/,/128.0/p' train.py

In [10]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u evaluate.py --run full_rgb --split test

2026-09-09 19:08:55.777085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788980935.799765     267 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788980935.807338     267 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788980935.825501     267 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788980935.825560     267 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788980935.825569     267 computation_placer.cc:177] computation placer alr